# Combining Data from Multiple Sources

### Merging datasets across formats (CSV and JSON), and the integration problems that come with it

**Moussa Doumbia, Ph.D.**
Department of Mathematics · Howard University

---

Run the cells in order. Every dataset is written to disk by the next cell, so the
notebook is self-contained — nothing to download.

**What you will be able to do by the end**

1. Load a CSV and a JSON file into `pandas` and describe how their shapes differ.
2. Diagnose the four failure modes that break a merge: mismatched **names**, mismatched **types**, mismatched **granularity**, and **duplicate keys**.
3. Choose the right join (`inner`, `left`, `outer`) for a stated question — and defend the choice.
4. Flatten nested JSON with `json_normalize`.
5. Use `validate=` and `indicator=True` to make a merge fail loudly instead of silently.

## 0. Setup — write the data files

These six files stand in for six real systems: a campus vendor spreadsheet, a
phone-app review feed, an athletics roster, a timing-system export, a transit
ridership report, and a weather API.

In [1]:
import json, os
from io import StringIO
import pandas as pd

pd.set_option("display.width", 110)
os.makedirs("data", exist_ok=True)

# ---- Example 1 : campus food trucks -------------------------------
open("data/trucks.csv", "w").write("""truck_id,truck_name,cuisine,lot
3,Mambo Sauce Mobile,Wings,Georgia Ave
7,Jollof Junction,West African,The Yard
12,Halal Bison,Halal,6th & Bryant
15,Sweet Potato Pi,Desserts,Founders Library
21,Bison Bowls,Salads,Blackburn
""")

reviews = [
    {"reviewId": 101, "truckId": "T-07", "stars": 5, "note": "Jollof worth the line"},
    {"reviewId": 102, "truckId": "T-03", "stars": 4, "note": "Extra mambo sauce"},
    {"reviewId": 103, "truckId": "T-07", "stars": 5, "note": "Plantain still hot"},
    {"reviewId": 104, "truckId": "T-12", "stars": 5, "note": "Best gyro on campus"},
    {"reviewId": 105, "truckId": "T-15", "stars": 3, "note": "Pie was cold"},
    {"reviewId": 106, "truckId": "T-03", "stars": 5, "note": "Fast at lunch rush"},
    {"reviewId": 107, "truckId": "T-12", "stars": 3, "note": "Long wait at noon"},
    {"reviewId": 108, "truckId": "T-99", "stars": 1, "note": "Truck never showed"},
]
json.dump(reviews, open("data/reviews.json", "w"), indent=2)

# ---- Example 2 : track & field ------------------------------------
open("data/roster.csv", "w").write("""athlete_id,athlete_name,class_year,squad
41,Amara Whitfield,Sophomore,Sprints
52,Devon Boakye,Senior,Jumps
63,Imani Carter,Freshman,Sprints
74,Kofi Mensah,Junior,Throws
""")

meet = {
    "meet_name": "Bison Invitational",
    "venue": "Greene Stadium",
    "athletes": [
        {"athlete_id": 41, "marks": [
            {"event": "100m", "mark": 11.62, "unit": "s", "place": 2},
            {"event": "200m", "mark": 23.98, "unit": "s", "place": 1}]},
        {"athlete_id": 52, "marks": [
            {"event": "Long Jump", "mark": 7.41, "unit": "m", "place": 1},
            {"event": "Triple Jump", "mark": 15.02, "unit": "m", "place": 3}]},
        {"athlete_id": 63, "marks": [
            {"event": "100m", "mark": 12.05, "unit": "s", "place": 5}]},
        {"athlete_id": 74, "marks": [
            {"event": "Shot Put", "mark": 17.83, "unit": "m", "place": 2},
            {"event": "Discus", "mark": 54.10, "unit": "m", "place": 1}]},
    ],
}
json.dump(meet, open("data/meet_results.json", "w"), indent=2)

# ---- Example 3 : Metro ridership vs weather -----------------------
open("data/ridership.csv", "w").write("""date,station,entries
03/02/2026,Shaw-Howard U,4120
03/02/2026,Georgia Ave-Petworth,5310
03/03/2026,Shaw-Howard U,4480
03/03/2026,Georgia Ave-Petworth,5602
03/04/2026,Shaw-Howard U,2110
03/04/2026,Georgia Ave-Petworth,2884
03/05/2026,Shaw-Howard U,4390
03/05/2026,Georgia Ave-Petworth,5501
""")

weather = [
    {"date": "2026-03-02", "tmax_f": 58, "precip_in": 0.00, "source": "station A"},
    {"date": "2026-03-03", "tmax_f": 61, "precip_in": 0.00, "source": "station A"},
    {"date": "2026-03-04", "tmax_f": 39, "precip_in": 1.27, "source": "station A"},
    {"date": "2026-03-04", "tmax_f": 39, "precip_in": 1.31, "source": "station B"},
    {"date": "2026-03-05", "tmax_f": 55, "precip_in": 0.02, "source": "station A"},
]
json.dump(weather, open("data/weather.json", "w"), indent=2)

print("pandas", pd.__version__)
print("files:", sorted(os.listdir("data")))

pandas 1.5.3
files: ['meet_results.json', 'reviews.json', 'ridership.csv', 'roster.csv', 'trucks.csv', 'weather.json']


## 1. Definitions

**Data integration.** Combining records that live in separate systems into one
table you can analyze. The hard part is almost never the code — it is deciding
*what counts as the same thing* in two systems that were never designed to talk
to each other.

**Key (join key).** The column, or set of columns, used to decide that a row on
the left and a row on the right describe the same entity. A merge is only as
trustworthy as its key.

**Grain (granularity).** What one row *means*. `trucks.csv` is one row per truck.
`reviews.json` is one row per review. Two tables at different grains cannot be
merged one-to-one — and pretending otherwise is the most common way a merge goes
quietly wrong.

**Join types.**

| Join | Keeps | Use it when |
|---|---|---|
| `inner` | rows whose key is in **both** | you only want matched records |
| `left` | **all** left rows, plus matches | the left table is your population |
| `right` | all right rows, plus matches | rare; just swap the tables instead |
| `outer` | everything from both sides | you are **auditing** what failed to match |

**Schema.** The names, types and meanings of the columns. CSV has one flat
schema per file. JSON does not have to be flat at all — it can nest lists inside
records, which is why it needs a flattening step before it can be merged.

**Concatenating vs. merging.** Merging adds **columns** (more facts about the
same entities). Concatenating adds **rows** (more entities, same facts). If you
are reaching for `merge` and the two tables have the same columns, you probably
want `pd.concat`.

### Why CSV and JSON fight each other

| | CSV | JSON |
|---|---|---|
| Shape | always a rectangle | can nest to any depth |
| Types | everything is text until parsed | has real types — but they are *declared*, not enforced |
| Missing values | empty cell | key may simply be absent |
| Keys | one flat column | may be nested, renamed, or a formatted code |

The practical consequence: a CSV column and a JSON field can carry the same
*idea* and still refuse to join, because one is the integer `7` and the other is
the string `"T-07"`.

---
## 2. Example 1 — The key that doesn't match

**The question.** Campus dining has a spreadsheet of food trucks. The student
app has a JSON feed of reviews. *Which truck has the best average rating?*

In [2]:
trucks  = pd.read_csv("data/trucks.csv")
reviews = pd.read_json("data/reviews.json")

display(trucks)
display(reviews)

,truck_id,truck_name,cuisine,lot
0,3,Mambo Sauce Mobile,Wings,Georgia Ave
1,7,Jollof Junction,West African,The Yard
2,12,Halal Bison,Halal,6th & Bryant
3,15,Sweet Potato Pi,Desserts,Founders Library
4,21,Bison Bowls,Salads,Blackburn


,reviewId,truckId,stars,note
0,101,T-07,5,Jollof worth the line
1,102,T-03,4,Extra mambo sauce
2,103,T-07,5,Plantain still hot
3,104,T-12,5,Best gyro on campus
4,105,T-15,3,Pie was cold
5,106,T-03,5,Fast at lunch rush
6,107,T-12,3,Long wait at noon
7,108,T-99,1,Truck never showed


In [3]:
# The two key columns have different NAMES and different TYPES.
print("trucks.truck_id :", trucks["truck_id"].dtype,  "| sample:", trucks["truck_id"].iloc[0])
print("reviews.truckId :", reviews["truckId"].dtype,  "| sample:", reviews["truckId"].iloc[0])

trucks.truck_id : int64 | sample: 3
reviews.truckId : object | sample: T-07


### The naive attempt

We line the two key columns up by name and ask pandas to merge. Watch what
happens.

In [4]:
try:
    trucks.merge(reviews, left_on="truck_id", right_on="truckId", how="inner")
except ValueError as e:
    print("ValueError:", e)

ValueError: You are trying to merge on int64 and object columns. If you wish to proceed you should use pd.concat


This is the **good** failure — pandas refused. The dangerous version of this bug
is when both columns are text, one padded (`"07"`) and one not (`"7"`): pandas
happily returns **zero rows** and reports no error at all. An empty result is not
proof that nothing matched; it is usually proof that your key is broken.

### The fix: normalize the key before merging

In [5]:
rev = reviews.rename(columns={"truckId": "truck_id"}).copy()
rev["truck_id"] = rev["truck_id"].str.replace("T-", "", regex=False).astype(int)

print(rev["truck_id"].dtype)
display(rev.head(3))

int64


,reviewId,truck_id,stars,note
0,101,7,5,Jollof worth the line
1,102,3,4,Extra mambo sauce
2,103,7,5,Plantain still hot


In [8]:
inner = trucks.merge(rev, on="truck_id", how="inner")
left  = trucks.merge(rev, on="truck_id", how="left")
outer = trucks.merge(rev, on="truck_id", how="outer", indicator=True)

print(f"trucks rows   : {len(trucks)}")
print(f"reviews rows  : {len(rev)}")
print(f"inner  -> {len(inner)}")
print(f"left   -> {len(left)}")
print(f"outer  -> {len(outer)}")

display(inner)
display(left)
display(outer)

trucks rows   : 5
reviews rows  : 8
inner  -> 7
left   -> 8
outer  -> 9


,truck_id,truck_name,cuisine,lot,reviewId,stars,note
0,3,Mambo Sauce Mobile,Wings,Georgia Ave,102,4,Extra mambo sauce
1,3,Mambo Sauce Mobile,Wings,Georgia Ave,106,5,Fast at lunch rush
2,7,Jollof Junction,West African,The Yard,101,5,Jollof worth the line
3,7,Jollof Junction,West African,The Yard,103,5,Plantain still hot
4,12,Halal Bison,Halal,6th & Bryant,104,5,Best gyro on campus
5,12,Halal Bison,Halal,6th & Bryant,107,3,Long wait at noon
6,15,Sweet Potato Pi,Desserts,Founders Library,105,3,Pie was cold


,truck_id,truck_name,cuisine,lot,reviewId,stars,note
0,3,Mambo Sauce Mobile,Wings,Georgia Ave,102.0,4.0,Extra mambo sauce
1,3,Mambo Sauce Mobile,Wings,Georgia Ave,106.0,5.0,Fast at lunch rush
2,7,Jollof Junction,West African,The Yard,101.0,5.0,Jollof worth the line
3,7,Jollof Junction,West African,The Yard,103.0,5.0,Plantain still hot
4,12,Halal Bison,Halal,6th & Bryant,104.0,5.0,Best gyro on campus
5,12,Halal Bison,Halal,6th & Bryant,107.0,3.0,Long wait at noon
6,15,Sweet Potato Pi,Desserts,Founders Library,105.0,3.0,Pie was cold
7,21,Bison Bowls,Salads,Blackburn,NaN,NaN,NaN


,truck_id,truck_name,cuisine,lot,reviewId,stars,note,_merge
0,3,Mambo Sauce Mobile,Wings,Georgia Ave,102.0,4.0,Extra mambo sauce,both
1,3,Mambo Sauce Mobile,Wings,Georgia Ave,106.0,5.0,Fast at lunch rush,both
2,7,Jollof Junction,West African,The Yard,101.0,5.0,Jollof worth the line,both
3,7,Jollof Junction,West African,The Yard,103.0,5.0,Plantain still hot,both
4,12,Halal Bison,Halal,6th & Bryant,104.0,5.0,Best gyro on campus,both
5,12,Halal Bison,Halal,6th & Bryant,107.0,3.0,Long wait at noon,both
6,15,Sweet Potato Pi,Desserts,Founders Library,105.0,3.0,Pie was cold,both
7,21,Bison Bowls,Salads,Blackburn,NaN,NaN,NaN,left_only
8,99,NaN,NaN,NaN,108.0,1.0,Truck never showed,right_only


### Audit the merge with `indicator=True`

Before trusting any join, look at what *didn't* match. This single habit catches
most integration bugs.

In [9]:
print(outer["_merge"].value_counts(), "\n")

print("In the spreadsheet but never reviewed:")
display(outer.loc[outer["_merge"] == "left_only", ["truck_id", "truck_name"]])

print("Reviewed but not in the spreadsheet (orphan key):")
display(outer.loc[outer["_merge"] == "right_only", ["truck_id", "reviewId", "stars", "note"]])

both          7
left_only     1
right_only    1
Name: _merge, dtype: int64 

In the spreadsheet but never reviewed:


,truck_id,truck_name
7,21,Bison Bowls


Reviewed but not in the spreadsheet (orphan key):


,truck_id,reviewId,stars,note
8,99,108.0,1.0,Truck never showed


Two real findings, neither visible in an `inner` join:

- **Bison Bowls** exists but has no reviews. An `inner` join would have deleted
  it, and a "trucks ranked by rating" report would silently omit a truck.
- Review 108 points at truck **99**, which does not exist. Someone reviewed a
  truck that was never registered — a data-entry problem worth reporting back to
  whoever owns the app.

In [40]:
answer = (inner.groupby(["truck_id", "truck_name"], as_index=False)
               .agg(n_reviews=("stars", "size"), avg_stars=("stars", "mean"))
               .sort_values("avg_stars", ascending=False))
display(answer)

,truck_id,truck_name,n_reviews,avg_stars
1,7,Jollof Junction,2,5.0
0,3,Mambo Sauce Mobile,2,4.5
2,12,Halal Bison,2,4.0
3,15,Sweet Potato Pi,1,3.0


**Answer.** Jollof Junction, 5.0 stars from 2 reviews.

But notice how thin that evidence is — every truck here has one or two reviews.
Reporting "best truck on campus" from two reviews is a statement about the data
collection, not about the food. The merge is correct; the conclusion still needs
a caveat.

---
## 3. Example 2 — Nested JSON, and the grain trap

**The question.** The athletics roster is a CSV. The timing system exports JSON
in which each athlete carries a *list* of marks. *How many events did each
athlete contest, and what was their best finish?*

In [41]:
meet = json.load(open("data/meet_results.json"))
print(json.dumps(meet, indent=2)[:420], "\n...")

{
  "meet_name": "Bison Invitational",
  "venue": "Greene Stadium",
  "athletes": [
    {
      "athlete_id": 41,
      "marks": [
        {
          "event": "100m",
          "mark": 11.62,
          "unit": "s",
          "place": 2
        },
        {
          "event": "200m",
          "mark": 23.98,
          "unit": "s",
          "place": 1
        }
      ]
    },
    {
      "athlete_id": 52,
      "mark 
...


This JSON is **not a rectangle**. `athletes` is a list, and inside each athlete
`marks` is another list. There is no sensible CSV that looks like this, which is
exactly why the two systems chose different formats.

### `json_normalize`: choose the grain you want

`record_path` names the list to explode into rows. `meta` names the fields from
the level *above* to carry down onto each row.

In [42]:
marks = pd.json_normalize(
    meet,
    record_path=["athletes", "marks"],      # one row per mark
    meta=[["athletes", "athlete_id"]],      # carry the id down
)
display(marks)
print(list(marks.columns))

,event,mark,unit,place,athletes.athlete_id
0,100m,11.62,s,2,41
1,200m,23.98,s,1,41
2,Long Jump,7.41,m,1,52
3,Triple Jump,15.02,m,3,52
4,100m,12.05,s,5,63
5,Shot Put,17.83,m,2,74
6,Discus,54.10,m,1,74


['event', 'mark', 'unit', 'place', 'athletes.athlete_id']


In [43]:
marks = marks.rename(columns={"athletes.athlete_id": "athlete_id"})
marks["athlete_id"] = marks["athlete_id"].astype(int)
display(marks)

,event,mark,unit,place,athlete_id
0,100m,11.62,s,2,41
1,200m,23.98,s,1,41
2,Long Jump,7.41,m,1,52
3,Triple Jump,15.02,m,3,52
4,100m,12.05,s,5,63
5,Shot Put,17.83,m,2,74
6,Discus,54.10,m,1,74


### One-to-many merge

The roster is **one row per athlete**. `marks` is **one row per mark**. The merge
is therefore one-to-many, and the result has more rows than the roster. That is
correct — but say it out loud before you merge, and let `validate=` hold you to
it.

In [44]:
roster = pd.read_csv("data/roster.csv")
display(roster)

full = roster.merge(marks, on="athlete_id", how="left", validate="one_to_many")
print(f"roster {len(roster)} rows  x  marks {len(marks)} rows  ->  merged {len(full)} rows")
display(full)

,athlete_id,athlete_name,class_year,squad
0,41,Amara Whitfield,Sophomore,Sprints
1,52,Devon Boakye,Senior,Jumps
2,63,Imani Carter,Freshman,Sprints
3,74,Kofi Mensah,Junior,Throws


roster 4 rows  x  marks 7 rows  ->  merged 7 rows


,athlete_id,athlete_name,class_year,squad,event,mark,unit,place
0,41,Amara Whitfield,Sophomore,Sprints,100m,11.62,s,2
1,41,Amara Whitfield,Sophomore,Sprints,200m,23.98,s,1
2,52,Devon Boakye,Senior,Jumps,Long Jump,7.41,m,1
3,52,Devon Boakye,Senior,Jumps,Triple Jump,15.02,m,3
4,63,Imani Carter,Freshman,Sprints,100m,12.05,s,5
5,74,Kofi Mensah,Junior,Throws,Shot Put,17.83,m,2
6,74,Kofi Mensah,Junior,Throws,Discus,54.10,m,1


Amara now occupies two rows. If you were to `sum` a per-athlete column at this
point — say, a scholarship amount stored on the roster — you would **double it**.
This is the single most expensive integration mistake, because the number it
produces looks plausible.

### Aggregate back to one row per athlete

In [45]:
summary = (full.groupby(["athlete_id", "athlete_name", "class_year", "squad"], as_index=False)
               .agg(events=("event", "count"),
                    best_place=("place", "min")))
display(summary)

,athlete_id,athlete_name,class_year,squad,events,best_place
0,41,Amara Whitfield,Sophomore,Sprints,2,1
1,52,Devon Boakye,Senior,Jumps,2,1
2,63,Imani Carter,Freshman,Sprints,1,5
3,74,Kofi Mensah,Junior,Throws,2,1


**Answer.** Amara, Devon and Kofi each contested 2 events; Imani contested 1.
Amara, Devon and Kofi each won at least one event (`best_place` = 1).

Note `best_place` uses `min`, because in track a *lower* place is better. Half of
data integration is remembering which direction "good" points in.

---
## 4. Example 3 — Duplicate keys, and dates that don't line up

**The question.** Metro publishes daily entries per station as CSV. A weather API
returns JSON. *Did the March 4 storm keep riders home?*

In [46]:
rides = pd.read_csv("data/ridership.csv")
wx    = pd.read_json("data/weather.json")

display(rides.head(4))
display(wx)

print("rides.date :", rides["date"].dtype, "| sample:", rides["date"].iloc[0])
print("wx.date    :", wx["date"].dtype,    "| sample:", wx["date"].iloc[0])

,date,station,entries
0,03/02/2026,Shaw-Howard U,4120
1,03/02/2026,Georgia Ave-Petworth,5310
2,03/03/2026,Shaw-Howard U,4480
3,03/03/2026,Georgia Ave-Petworth,5602


,date,tmax_f,precip_in,source
0,2026-03-02,58,0.00,station A
1,2026-03-03,61,0.00,station A
2,2026-03-04,39,1.27,station A
3,2026-03-04,39,1.31,station B
4,2026-03-05,55,0.02,station A


rides.date : str | sample: 03/02/2026
wx.date    : datetime64[us] | sample: 2026-03-02 00:00:00


`read_json` recognized the ISO format `2026-03-04` and parsed it to a real
timestamp. `read_csv` did **not** recognize `03/04/2026` and left it as text.
Merging text against timestamps matches nothing.

Parse both sides explicitly. Always pass `format=` for an ambiguous layout —
`03/04/2026` is March 4 in the US and 3 April almost everywhere else, and
guessing wrong shifts your whole analysis by days.

In [47]:
rides["date"] = pd.to_datetime(rides["date"], format="%m/%d/%Y")
wx["date"]    = pd.to_datetime(wx["date"])
print(rides["date"].dtype, "|", wx["date"].dtype)

datetime64[us] | datetime64[us]


### The duplicate-key trap

Before merging, ask whether the key is unique on the side you expect it to be.

In [48]:
print("weather rows:", len(wx), " unique dates:", wx["date"].nunique())
display(wx[wx.duplicated("date", keep=False)])

weather rows: 5  unique dates: 4


,date,tmax_f,precip_in,source
2,2026-03-04,39,1.27,station A
3,2026-03-04,39,1.31,station B


Two weather stations both reported March 4. Nobody made an error — the feed
simply has a finer grain than we assumed. Now watch the damage.

In [49]:
bad = rides.merge(wx, on="date", how="left")
print(f"ridership rows: {len(rides)}  ->  after merge: {len(bad)}")
display(bad[bad["date"] == "2026-03-04"])

ridership rows: 8  ->  after merge: 10


,date,station,entries,tmax_f,precip_in,source
4,2026-03-04,Shaw-Howard U,2110,39,1.27,station A
5,2026-03-04,Shaw-Howard U,2110,39,1.31,station B
6,2026-03-04,Georgia Ave-Petworth,2884,39,1.27,station A
7,2026-03-04,Georgia Ave-Petworth,2884,39,1.31,station B


March 4's ridership is now counted **twice**. Any daily total computed from this
table is wrong for that day, and nothing in the output says so.

### Make the merge fail loudly

`validate="many_to_one"` asserts that the key is unique on the right. When the
assertion is false, pandas raises instead of silently duplicating.

In [50]:
try:
    rides.merge(wx, on="date", how="left", validate="many_to_one")
except Exception as e:
    print(type(e).__name__, "\n", e)

MergeError 
 Merge keys are not unique in right dataset; not a many-to-one merge

Duplicates in right:
       date
2026-03-04 ...


### Fix the grain, then merge

Decide explicitly what one weather row per day should mean. Here we keep
station A as the official reading; averaging the two stations would be equally
defensible — what matters is that the choice is deliberate and recorded.

In [19]:
wx_daily = wx.sort_values("source").drop_duplicates("date", keep="first")

clean = rides.merge(wx_daily, on="date", how="left", validate="many_to_one")
print("rows:", len(clean))
display(clean)

rows: 8


,date,station,entries,tmax_f,precip_in,source
0,2026-03-02,Shaw-Howard U,4120,58,0.00,station A
1,2026-03-02,Georgia Ave-Petworth,5310,58,0.00,station A
2,2026-03-03,Shaw-Howard U,4480,61,0.00,station A
3,2026-03-03,Georgia Ave-Petworth,5602,61,0.00,station A
4,2026-03-04,Shaw-Howard U,2110,39,1.27,station A
5,2026-03-04,Georgia Ave-Petworth,2884,39,1.27,station A
6,2026-03-05,Shaw-Howard U,4390,55,0.02,station A
7,2026-03-05,Georgia Ave-Petworth,5501,55,0.02,station A


In [20]:
daily = (clean.groupby("date", as_index=False)
              .agg(total_entries=("entries", "sum"),
                   tmax_f=("tmax_f", "first"),
                   precip_in=("precip_in", "first")))
display(daily)

dry  = daily.loc[daily["precip_in"] < 0.1, "total_entries"].mean()
rain = daily.loc[daily["date"] == "2026-03-04", "total_entries"].iloc[0]
print(f"average dry-day entries : {dry:,.0f}")
print(f"March 4 (1.27 in rain)  : {rain:,.0f}")
print(f"change                  : {(rain/dry - 1)*100:.1f}%")

,date,total_entries,tmax_f,precip_in
0,2026-03-02,9430,58,0.00
1,2026-03-03,10082,61,0.00
2,2026-03-04,4994,39,1.27
3,2026-03-05,9891,55,0.02


average dry-day entries : 9,801
March 4 (1.27 in rain)  : 4,994
change                  : -49.0%


**Answer.** Ridership on the storm day was about **49% below** the average dry
day.

And now the honest caveat: this is **four days** of data at **two stations**. The
drop is large enough to be interesting and nowhere near enough to establish that
rain causes it — March 4 might also have been a holiday, a snow closure, or a
service outage. A merge gives you a table, not a conclusion.

---
## 5. The checklist

Run this before every merge. It takes a minute and saves an afternoon.

1. **What is one row?** Say the grain of each table out loud.
2. **What is the key?** Does it mean the same thing on both sides?
3. **Do the types match?** `df[key].dtype` on both.
4. **Is the key unique** where you think it is? `df[key].is_unique`
5. **Which join** answers the question — and which rows am I prepared to lose?
6. **Assert it:** `validate="one_to_one" | "one_to_many" | "many_to_one"`
7. **Audit it:** `indicator=True`, then look at `left_only` and `right_only`.
8. **Count rows before and after.** A surprise means a duplicate key.

---
# Practice problems

Write your answer in the empty cell under each problem. Solutions follow at the
end — try each one first.

### Problem 1 — Audit before you trust *(food-truck data)*

1. Using an `outer` merge with `indicator=True`, produce a table of every review
   whose `truck_id` does not appear in `trucks.csv`.
2. Recompute the average-stars ranking, but keep only trucks with **at least 2
   reviews**.
3. In one sentence: why would a `left` join be the wrong choice for part 2?

In [21]:
# your work for Problem 1

### Problem 2 — Which squad won the meet? *(track data)*

Using `full` (roster merged with marks):

1. Count first-place finishes (`place == 1`) per `squad`.
2. Award 5 points for 1st, 3 for 2nd, 1 for 3rd, 0 otherwise, and rank the squads
   by total points.
3. Why can you not answer part 2 from `roster` and `marks` separately?

In [22]:
# your work for Problem 2

### Problem 3 — Break it on purpose *(Metro data)*

1. Merge `rides` with the **unfixed** `wx` and compute daily totals. By how much
   does March 4 differ from the correct figure?
2. Instead of dropping a station, **average** the two March 4 readings and redo
   the merge. Does the conclusion change?
3. Which of the two repairs would you defend to a transit planner, and why?

In [23]:
# your work for Problem 3

### Problem 4 — Two sources disagree

A directory export (CSV) has stale emails; a registrar feed (JSON) has updates
for some students. Build one table with the **best available** email for each
student: use the registrar value when present, otherwise the directory value,
and flag anyone with no email at all.

*Hint: `merge`, then `combine_first` or `fillna`.*

In [24]:
directory = pd.DataFrame({
    "student_id": [1001, 1002, 1003, 1004, 1005],
    "name":       ["Ayana Brooks", "Marcus Hall", "Nia Okonkwo", "Tariq Reed", "Zoe Adjei"],
    "email":      ["ayana@old.edu", None, "nia@old.edu", None, "zoe@old.edu"],
})

registrar = pd.read_json(StringIO(json.dumps([
    {"student_id": 1002, "email": "marcus.hall@bison.edu"},
    {"student_id": 1003, "email": "nia.okonkwo@bison.edu"},
    {"student_id": 1006, "email": "ghost@bison.edu"},
])))

display(directory); display(registrar)

,student_id,name,email
0,1001,Ayana Brooks,ayana@old.edu
1,1002,Marcus Hall,NaN
2,1003,Nia Okonkwo,nia@old.edu
3,1004,Tariq Reed,NaN
4,1005,Zoe Adjei,zoe@old.edu


,student_id,email
0,1002,marcus.hall@bison.edu
1,1003,nia.okonkwo@bison.edu
2,1006,ghost@bison.edu


In [25]:
# your work for Problem 4

### Problem 5 — Merge or concatenate?

Two halls ran the same event survey. Blackburn exported CSV; Armour J. Blair
returned JSON with different field names. Produce **one** table of all responses
with columns `hall`, `event`, `attendees`.

Then answer: is this a `merge` or a `concat`, and how did you know?

In [26]:
blackburn = pd.read_csv(StringIO(
    "event,attendees\nOpen Mic,84\nCareer Fair,210\nGame Night,57\n"))

blair = pd.read_json(StringIO(json.dumps([
    {"eventName": "Open Mic",    "headCount": 61},
    {"eventName": "Study Jam",   "headCount": 133},
    {"eventName": "Game Night",  "headCount": 45},
])))

display(blackburn); display(blair)

,event,attendees
0,Open Mic,84
1,Career Fair,210
2,Game Night,57


,eventName,headCount
0,Open Mic,61
1,Study Jam,133
2,Game Night,45


In [27]:
# your work for Problem 5

---
# Solutions

### Solution 1

In [28]:
# 1. orphan reviews
orphans = outer.loc[outer["_merge"] == "right_only",
                    ["truck_id", "reviewId", "stars", "note"]]
display(orphans)

# 2. trucks with >= 2 reviews
rank = (inner.groupby(["truck_id", "truck_name"], as_index=False)
             .agg(n_reviews=("stars", "size"), avg_stars=("stars", "mean")))
display(rank[rank["n_reviews"] >= 2].sort_values("avg_stars", ascending=False))

# 3. A left join keeps trucks with NO reviews, giving them avg_stars = NaN
#    (or 0 if filled) and n_reviews = 1 from the all-null row. The >= 2 filter
#    would remove them anyway -- but the row count and any mean computed over
#    all trucks would already be wrong. Inner is the honest choice here.

,truck_id,reviewId,stars,note
8,99,108.0,1.0,Truck never showed


,truck_id,truck_name,n_reviews,avg_stars
1,7,Jollof Junction,2,5.0
0,3,Mambo Sauce Mobile,2,4.5
2,12,Halal Bison,2,4.0


### Solution 2

In [29]:
# 1. first places per squad
firsts = (full[full["place"] == 1]
          .groupby("squad", as_index=False)
          .agg(first_places=("place", "size")))
display(firsts)

# 2. points
points_map = {1: 5, 2: 3, 3: 1}
full2 = full.copy()
full2["points"] = full2["place"].map(points_map).fillna(0).astype(int)

standings = (full2.groupby("squad", as_index=False)
                  .agg(points=("points", "sum"), athletes=("athlete_id", "nunique"))
                  .sort_values("points", ascending=False))
display(standings)

# 3. `squad` lives only in the CSV roster; `place` lives only in the JSON marks.
#    Neither table can answer the question alone -- that is precisely what the
#    merge is for.

,squad,first_places
0,Jumps,1
1,Sprints,1
2,Throws,1


,squad,points,athletes
1,Sprints,8,2
2,Throws,8,1
0,Jumps,6,1


### Solution 3

In [30]:
# 1. the broken version
bad_daily = (rides.merge(wx, on="date", how="left")
                  .groupby("date", as_index=False)
                  .agg(total_entries=("entries", "sum")))
comparison = bad_daily.merge(daily[["date", "total_entries"]],
                             on="date", suffixes=("_broken", "_correct"))
comparison["overstated_by"] = (comparison["total_entries_broken"]
                               - comparison["total_entries_correct"])
display(comparison)

# 2. average the two stations instead of dropping one
wx_avg = (wx.groupby("date", as_index=False)
            .agg(tmax_f=("tmax_f", "mean"), precip_in=("precip_in", "mean")))
clean2 = rides.merge(wx_avg, on="date", how="left", validate="many_to_one")
daily2 = (clean2.groupby("date", as_index=False)
                .agg(total_entries=("entries", "sum"),
                     precip_in=("precip_in", "first")))
display(daily2)

dry2  = daily2.loc[daily2["precip_in"] < 0.1, "total_entries"].mean()
rain2 = daily2.loc[daily2["date"] == "2026-03-04", "total_entries"].iloc[0]
print(f"change with averaged weather: {(rain2/dry2 - 1)*100:.1f}%")

# 3. The ridership totals are identical either way -- only the precip figure
#    moves (1.27 vs 1.29 in). Averaging uses all the evidence and does not
#    require picking a favorite station, so it is the easier choice to defend.
#    Dropping a station is defensible ONLY if one station is the official gauge,
#    and then you should say so in writing.

,date,total_entries_broken,total_entries_correct,overstated_by
0,2026-03-02,9430,9430,0
1,2026-03-03,10082,10082,0
2,2026-03-04,9988,4994,4994
3,2026-03-05,9891,9891,0


,date,total_entries,precip_in
0,2026-03-02,9430,0.00
1,2026-03-03,10082,0.00
2,2026-03-04,4994,1.29
3,2026-03-05,9891,0.02


change with averaged weather: -49.0%


### Solution 4

In [31]:
merged = directory.merge(registrar, on="student_id", how="left",
                         suffixes=("_dir", "_reg"))

# registrar wins where present, directory fills the gaps
merged["email"] = merged["email_reg"].combine_first(merged["email_dir"])
merged["no_email"] = merged["email"].isna()

display(merged[["student_id", "name", "email_dir", "email_reg", "email", "no_email"]])

# Student 1006 is in the registrar feed but not the directory -- a LEFT join
# drops them silently. Check for that separately:
missing = registrar.loc[~registrar["student_id"].isin(directory["student_id"])]
print("in registrar but not in directory:")
display(missing)

,student_id,name,email_dir,email_reg,email,no_email
0,1001,Ayana Brooks,ayana@old.edu,NaN,ayana@old.edu,False
1,1002,Marcus Hall,NaN,marcus.hall@bison.edu,marcus.hall@bison.edu,False
2,1003,Nia Okonkwo,nia@old.edu,nia.okonkwo@bison.edu,nia.okonkwo@bison.edu,False
3,1004,Tariq Reed,NaN,NaN,NaN,True
4,1005,Zoe Adjei,zoe@old.edu,NaN,zoe@old.edu,False


in registrar but not in directory:


,student_id,email
2,1006,ghost@bison.edu


### Solution 5

In [32]:
b1 = blackburn.assign(hall="Blackburn")
b2 = (blair.rename(columns={"eventName": "event", "headCount": "attendees"})
           .assign(hall="Armour J. Blair"))

all_responses = pd.concat([b1, b2], ignore_index=True)[["hall", "event", "attendees"]]
display(all_responses)

# It is a CONCAT. The two tables describe the SAME kind of thing (an event with
# an attendance count) at the SAME grain -- they are more ROWS, not more columns.
# A merge would have been right only if one file held facts the other lacked,
# e.g. a room capacity per event.

# Follow-up: events that ran in both halls
display(all_responses.pivot_table(index="event", columns="hall",
                                  values="attendees", aggfunc="sum"))

,hall,event,attendees
0,Blackburn,Open Mic,84
1,Blackburn,Career Fair,210
2,Blackburn,Game Night,57
3,Armour J. Blair,Open Mic,61
4,Armour J. Blair,Study Jam,133
5,Armour J. Blair,Game Night,45


hall,Armour J. Blair,Blackburn
event,,
Career Fair,NaN,210.0
Game Night,45.0,57.0
Open Mic,61.0,84.0
Study Jam,133.0,NaN


---

*Combining Data from Multiple Sources* · Moussa Doumbia, Ph.D. ·
Department of Mathematics · Howard University